# Shor's Algorithm

Factor 15 using the quantum order-finding subroutine. The quantum circuit applies controlled modular exponentiation, then inverse QFT to extract the period.

In [ ]:
import pennylane as qml
import numpy as np
from math import gcd

## Circuit setup

4 precision qubits + 4 target qubits for mod-18 arithmetic.

In [ ]:
N = 15
N_PREC = 4
N_TARGET = 4
N_TOTAL = N_PREC + N_TARGET
dev = qml.device("default.qubit", wires=N_TOTAL)

@qml.qnode(dev)
def shor_circuit(a):
    qml.PauliX(wires=N_PREC)
    qml.Hadamard(wires=range(N_PREC))
    for k in range(N_PREC):
        power = 2**k
        a_pow = pow(a, power, N)
        for i in range(N_TARGET):
            qml.ctrl(qml.Pow(qml.PauliX(wires=N_PREC + i), a_pow),
                     control=k)
    qml.adjoint(qml.QFT(wires=list(range(N_PREC))))
    return qml.probs(wires=range(N_PREC))

print("Circuit for a = 7:")
print(qml.draw(shor_circuit)(7))

## Order finding

In [ ]:
a = 7
probs = shor_circuit(a)
measured = np.argmax(probs)
phase = measured / (2**N_PREC)
print(f"a = {a}")
print(f"Measured: |{measured:0{N_PREC}b}\u27e9  phase \u2248 {phase:.4f}")

if measured != 0:
    r = int(round(1 / phase))
    if pow(a, r, N) == 1:
        print(f"Recovered order: r = {r}")
        print(f"Verify: {a}\u00d7\u2071\u00b2\u00b9 mod {N} = {pow(a, r, N)}")

## Factorization

In [ ]:
def factor(n, rng):
    while True:
        a = int(rng.integers(2, n))
        g = gcd(a, n)
        if g != 1:
            return g, n // g
        probs = shor_circuit(a)
        measured = np.argmax(probs)
        if measured == 0:
            continue
        phase = measured / (2**N_PREC)
        r = int(round(1 / phase))
        if r % 2 == 0 and pow(a, r, n) == 1:
            x = pow(a, r // 2, n)
            if x not in (1, n - 1):
                f1 = gcd(x - 1, n)
                f2 = gcd(x + 1, n)
                if 1 < f1 < n:
                    return f1, n // f1
                if 1 < f2 < n:
                    return f2, n // f2

rng = np.random.default_rng(42)
p, q = factor(N, rng)
print(f"{N} = {p} \u00d7 {q}")
print(f"Verify: {p} \u00d7 {q} = {p * q}")